# Exploratory Data Analysis

## Objectives

- Verify dataset integrity and structure.
- Understand the annotation format.
- Analyze defect class distribution.
- Examine normal and defective images.
- Decode and visualize segmentation masks.
- Identify challenges for model training.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

DATA_DIR = Path("/Volumes/YourSSD/datasets/severstal")

TRAIN_DIR = DATA_DIR / "train_images"
TEST_DIR = DATA_DIR / "test_images"
TRAIN_CSV = DATA_DIR / "train.csv"

SEED = 42

In [2]:
assert DATA_DIR.is_dir(), "Dataset directory not found"
assert TRAIN_DIR.is_dir(), "Training image directory not found"
assert TRAIN_CSV.is_file(), "Training annotations not found"

train_images = sorted(TRAIN_DIR.glob("*.jpg"))

df = pd.read_csv(TRAIN_CSV)

print(f"Training images: {len(train_images):,}")
print(f"Annotation rows: {len(df):,}")
print(f"Annotation columns: {df.columns.tolist()}")

display(df.head())

Training images: 25,136
Annotation rows: 7,095
Annotation columns: ['ImageId', 'ClassId', 'EncodedPixels']


,ImageId,ClassId,EncodedPixels
0,0002cc93b.jpg,1,29102 12 29346 24 29602 24 29858 24 30114 24 3...
1,0007a71bf.jpg,3,18661 28 18863 82 19091 110 19347 110 19603 11...
2,000a4bcdd.jpg,1,37607 3 37858 8 38108 14 38359 20 38610 25 388...
3,000f6bf48.jpg,4,131973 1 132228 4 132483 6 132738 8 132993 11 ...
4,0014fce06.jpg,3,229501 11 229741 33 229981 55 230221 77 230468...


## 1.1 Training Image Verification

Verify the number of training images and identify hidden
system-generated files that should be excluded from the dataset.

This ensures accurate sample counts and prevents invalid files
from being loaded during model training.

In [3]:
all_image_files = sorted(TRAIN_DIR.glob("*.jpg"))

hidden_files = [
    path for path in all_image_files
    if path.name.startswith(".")
]

train_images = [
    path for path in all_image_files
    if not path.name.startswith(".")
]

print(f"Total JPG files: {len(all_image_files):,}")
print(f"Hidden JPG files: {len(hidden_files):,}")
print(f"Actual training images: {len(train_images):,}")

print("\nSample hidden files:")
print([path.name for path in hidden_files[:5]])

Total JPG files: 25,136
Hidden JPG files: 12,568
Actual training images: 12,568

Sample hidden files:
['._0002cc93b.jpg', '._00031f466.jpg', '._000418bfc.jpg', '._000789191.jpg', '._0007a71bf.jpg']


## 1.2 Annotation Structure

The annotation dataset contains image identifiers, defect class
labels, and run-length encoded segmentation masks.

Because an image may contain multiple defect classes,
the number of annotation rows is not necessarily equal
to the number of unique defective images.

In [4]:
print(f"Annotation rows: {len(df):,}")
print(f"Unique annotated images: {df['ImageId'].nunique():,}")
print(f"Defect classes: {sorted(df['ClassId'].unique())}")
print(f"Missing masks: {df['EncodedPixels'].isna().sum():,}")

print("\nDuplicate image-class pairs:")
print(df.duplicated(subset=["ImageId", "ClassId"]).sum())

Annotation rows: 7,095
Unique annotated images: 6,666
Defect classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Missing masks: 0

Duplicate image-class pairs:
0


In [5]:
# Validate annotation-image consistency

image_ids = {path.name for path in train_images}
annotated_ids = set(df["ImageId"])

missing_images = annotated_ids - image_ids

print(f"Annotations with missing images: {len(missing_images)}")

Annotations with missing images: 0
